In [ ]:
#step1
# 1. Font configuration
!pip install -q japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import files
from scipy.interpolate import PchipInterpolator

# 2. Upload rainfall file
print("Select the Yatsushiro 10-minute rainfall Excel file (.xlsx):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 3. Load Sheet2 (second worksheet)
df = pd.read_excel(file_name, sheet_name=1)
print(f"Columns in the loaded worksheet: {list(df.columns)}")

# Automatically identify the rainfall column
rain_col = None
for col in df.columns:
  if any(keyword in str(col).lower() for keyword in ["雨量", "降雨", "rain"]):
    rain_col = col
    break

if rain_col is None:
  max_valid_count = 0
  for col in df.columns:
    s = pd.to_numeric(df[col], errors="coerce")
    valid_count = (s > 0).sum()
    if valid_count > max_valid_count:
      max_valid_count = valid_count
      rain_col = col

print(f"Selected rainfall column: {rain_col}")

# Create the 10-minute rainfall array
r_10min = (
    pd.to_numeric(df[rain_col], errors="coerce").fillna(0.0).values.astype(float)
)

# 4. PCHIP interpolation of cumulative rainfall
cum_10min = np.insert(np.cumsum(r_10min), 0, 0.0)
t_10min = np.arange(len(cum_10min)) * 10
t_1min = np.arange(0, t_10min[-1] + 1, 1)

pchip = PchipInterpolator(t_10min, cum_10min)
cum_1min = pchip(t_1min)

# Calculate 1-minute rainfall intensity [mm/min]
r_1min = np.diff(cum_1min)
r_1min = np.clip(r_1min, 0.0, None)

# 5. Diagnostic plot
plt.figure(figsize=(10, 4))
plt.plot(r_1min, label="PCHIP 1-minute rainfall r_1min [mm/min]", color="blue")
plt.xlabel("Time [min]")
plt.ylabel("Rainfall intensity [mm/min]")
plt.title(f"Rainfall intensity hyetograph ({file_name} [Sheet2] -> PCHIP 1-minute data)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total rainfall: {np.sum(r_1min):.2f} mm")
print(f"Maximum 1-minute rainfall: {np.max(r_1min):.4f} mm/min")

In [ ]:

# ============================================================
# Step 2: Schoolyard detention model
# Area: 4,000 m²; maximum storage depth: 20 cm
# Compare orifice diameters of 5, 10, 15, and 20 cm
# Use r_1min from Step 1 without modification
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Model parameters
# ------------------------------------------------------------

AREA = 4000.0         # Schoolyard area [m²]
H_MAX = 0.20          # Maximum storage depth [m]
CD = 0.60             # Discharge coefficient
G = 9.81              # Gravitational acceleration [m/s²]

DIAMETERS_CM = [5, 10, 15, 20]

DT = 5.0              # Internal time step [s]
MAX_STORAGE = AREA * H_MAX

rain = np.asarray(r_1min, dtype=float)

if not np.all(np.isfinite(rain)) or np.any(rain < 0):
    raise ValueError("r_1min contains invalid values")

print(f"Schoolyard area: {AREA:.0f} m²")
print(f"Maximum storage depth: {H_MAX*100:.0f} cm")
print(f"Maximum storage capacity: {MAX_STORAGE:.0f} m³")
print(f"Total rainfall: {rain.sum():.2f} mm")


# ------------------------------------------------------------
# 2. Calculate discharge through a circular orifice
# Set the bottom of the orifice at schoolyard ground level
# Account for partial submergence of the opening
# ------------------------------------------------------------

def orifice_flow(h, diameter_cm):

    D = diameter_cm / 100.0
    R = D / 2.0

    if h <= 0:
        return 0.0

    # Divide the opening into horizontal strips
    n = 100
    dz = D / n
    z = (np.arange(n) + 0.5) * dz

    width = 2 * np.sqrt(
        np.maximum(R**2 - (z-R)**2, 0.0)
    )

    head = np.maximum(h-z, 0.0)

    Q = CD * np.sum(
        width * dz * np.sqrt(2*G*head)
    )

    return Q  # [m³/s]


# ------------------------------------------------------------
# 3. Schoolyard detention calculation
# ------------------------------------------------------------

def school_storage(rain, diameter_cm):

    n = len(rain)

    depth = np.zeros(n+1)
    outflow = np.zeros(n)
    overflow = np.zeros(n)

    storage = 0.0
    substeps = int(round(60.0 / DT))

    for i in range(n):

        vout_total = 0.0
        vover_total = 0.0

        # Assume constant rainfall intensity within each 1-minute step
        rain_volume_step = (
            rain[i] / 1000.0 * AREA / substeps
        )

        for j in range(substeps):

            storage += rain_volume_step

            # Current storage depth
            h = storage / AREA

            # Orifice discharge
            Q = orifice_flow(h, diameter_cm)
            vout = min(Q * DT, storage)
            storage -= vout

            # Excess above maximum capacity is overflow
            vover = max(storage - MAX_STORAGE, 0.0)
            storage -= vover

            vout_total += vout
            vover_total += vover

        depth[i+1] = storage / AREA

        # 1-minute mean discharge [m³/s]
        outflow[i] = vout_total / 60.0
        overflow[i] = vover_total / 60.0

    return depth, outflow, overflow


# ------------------------------------------------------------
# 4. Compute each orifice diameter
# ------------------------------------------------------------

time = np.arange(len(rain))
results = []
hydrographs = {}
depths = {}

for D in DIAMETERS_CM:

    depth, qout, qover = school_storage(rain, D)

    # Runoff depth [mm/min]
    runoff = (qout + qover) * 60 / AREA * 1000

    hydrographs[D] = runoff
    depths[D] = depth

    peak = np.max(runoff)
    reduction = (
        (1 - peak / np.max(rain)) * 100
        if np.max(rain) > 0 else 0.0
    )

    results.append({
        "Orifice diameter (cm)": D,
        "Maximum depth (cm)": np.max(depth)*100,
        "Maximum runoff intensity (mm/min)": peak,
        "Peak reduction (%)": reduction,
        "Overflow volume (m³)": np.sum(qover)*60,
        "Overflow duration (min)": np.sum(qover > 1e-12),
        "Final stored volume (m³)": depth[-1]*AREA
    })


# ------------------------------------------------------------
# 5. Runoff hydrograph
# ------------------------------------------------------------

plt.figure(figsize=(15, 6))

plt.plot(
    time, rain,
    color="black", linestyle="--",
    linewidth=2.2,
    label="No facility"
)

for D in DIAMETERS_CM:
    plt.plot(
        time, hydrographs[D],
        linewidth=2,
        label=f"Orifice diameter {D} cm"
    )

plt.xlabel("Time (min)", fontsize=19)
plt.ylabel("Runoff intensity (mm/min)", fontsize=19)
plt.title("Runoff hydrographs with schoolyard detention", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 6. Storage depth
# ------------------------------------------------------------

plt.figure(figsize=(15, 6))

for D in DIAMETERS_CM:
    plt.plot(
        np.arange(len(depths[D])),
        depths[D]*100,
        linewidth=2,
        label=f"Orifice diameter {D} cm"
    )

plt.axhline(
    H_MAX*100,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Maximum storage depth"
)

plt.xlabel("Time (min)", fontsize=19)
plt.ylabel("Storage depth (cm)", fontsize=19)
plt.title("Schoolyard storage depth over time", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. Comparison table
# ------------------------------------------------------------

result_df = pd.DataFrame(results)

print("\n===== Comparison by orifice diameter =====")
print(result_df.round(3).to_string(index=False))

best = result_df.loc[
    result_df["Maximum runoff intensity (mm/min)"].idxmin()
]

print("\n===== Orifice diameter giving the lowest peak for this rainfall event =====")
print(f"Orifice diameter: {best['Orifice diameter (cm)']:.0f} cm")
print(
    f"Maximum runoff intensity:"
    f"{best['Maximum runoff intensity (mm/min)']:.4f} mm/min"
)
print(
    f"Peak reduction:"
    f"{best['Peak reduction (%)']:.1f} %"
)
print(
    f"Overflow volume:{best['Overflow volume (m³)']:.1f} m³"
)


In [ ]:
# ============================================================
# Step 3: Schoolyard detention envelope analysis
# 1 Runoff intensity envelope
# 2 Delta-rho envelope and original rainfall envelope
# Y-axis: mm/h; x-axis: window duration (min)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Time window
MAX_WINDOW = min(180, len(rain))
windows = np.arange(1, MAX_WINDOW + 1)

# 2. Envelope calculation function
def calc_envelope(wave, windows):
    wave = np.asarray(wave, dtype=float)
    cumulative = np.insert(np.cumsum(wave), 0, 0.0)

    envelope = np.zeros(len(windows))

    for i, T in enumerate(windows):
        moving_average = (
            cumulative[T:] - cumulative[:-T]
        ) / T
        envelope[i] = np.max(moving_average)

    return envelope

# 3. Convert the envelope to mm/h
E_natural = calc_envelope(rain, windows) * 60

E_storage = {}
delta_rho = {}

for D in DIAMETERS_CM:
    E_storage[D] = calc_envelope(
        hydrographs[D], windows
    ) * 60

    delta_rho[D] = E_natural - E_storage[D]

colors = {
    5: "blue",
    10: "green",
    15: "orange",
    20: "red"
}

# ------------------------------------------------------------
# 4. Plot 1: Runoff intensity envelope
# ------------------------------------------------------------

plt.figure(figsize=(15, 7))

plt.plot(
    windows, E_natural,
    color="black",
    linewidth=3,
    label="No facility (rainfall)"
)

for D in DIAMETERS_CM:
    plt.plot(
        windows, E_storage[D],
        color=colors.get(D, None),
        linewidth=2.2,
        label=f"Orifice diameter {D} cm"
    )

plt.xlabel("Time window T (min)", fontsize=19)
plt.ylabel("Maximum mean runoff intensity (mm/h)", fontsize=19)
plt.title("Schoolyard detention: runoff envelope", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(True, alpha=0.3)
plt.xlim(0, MAX_WINDOW)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Plot 2: Δρ＋Original rainfall envelope
# ------------------------------------------------------------

plt.figure(figsize=(15, 7))

# Original rainfall envelope
plt.plot(
    windows, E_natural,
    color="black",
    linestyle="--",
    linewidth=3,
    label="Original rainfall envelope"
)

# Δρ
for D in DIAMETERS_CM:
    plt.plot(
        windows, delta_rho[D],
        color=colors.get(D, None),
        linewidth=2.2,
        label=f"Δρ: Orifice diameter {D} cm"
    )

plt.axhline(
    0, color="gray",
    linestyle=":", linewidth=1
)

plt.xlabel("Time window T (min)", fontsize=19)
plt.ylabel("Rainfall intensity / Δρ (mm/h)", fontsize=19)
plt.title("Schoolyard detention: Δρ and rainfall envelope", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(True, alpha=0.3)
plt.xlim(0, MAX_WINDOW)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Results table
# ------------------------------------------------------------

envelope_df = pd.DataFrame({
    "Time window (min)": windows,
    "No facility (mm/h)": E_natural
})

for D in DIAMETERS_CM:
    envelope_df[f"Diameter {D}cm_Runoff intensity (mm/h)"] = E_storage[D]
    envelope_df[f"Diameter {D}cm_Δρ(mm/h)"] = delta_rho[D]

display(envelope_df.head(10))

In [ ]:
# ============================================================
# Step 4: Yashikirin (traditional homestead woodland) infiltration and canopy interception model
# 1 Runoff depthHydrograph
# 2 Delta-rho envelope and original rainfall envelope
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Model parameters
# ------------------------------------------------------------

FOREST_AREA = 1200.0       # Woodland area [m²]
INTERCEPTION = 0.10        # Canopy interception fraction 10%
INFILTRATION_RATES = [50, 75, 100]  # Infiltration capacity [mm/h]

rain = np.asarray(r_1min, dtype=float)  # [mm/min]

if not np.all(np.isfinite(rain)) or np.any(rain < 0):
    raise ValueError("r_1min contains invalid values")

time = np.arange(len(rain))

# ------------------------------------------------------------
# 2. Woodland runoff calculation
# Intercept 10% of rainfall; subtract infiltration from the remainder
# ------------------------------------------------------------

def forest_runoff(rain_mm_min, infiltration_mm_h):

    # Rainfall after canopy interception
    effective_rain = rain_mm_min * (1 - INTERCEPTION)

    # Convert infiltration capacity to mm/min
    infiltration_mm_min = infiltration_mm_h / 60.0

    # Infiltration
    infiltration = np.minimum(
        effective_rain, infiltration_mm_min
    )

    # Surface runoff depth [mm/min]
    runoff = np.maximum(
        effective_rain - infiltration, 0.0
    )

    return runoff, infiltration


# ------------------------------------------------------------
# 3. Runoff calculation for each infiltration capacity
# ------------------------------------------------------------

forest_hydrographs = {}
forest_results = []

for f in INFILTRATION_RATES:

    runoff, infiltration = forest_runoff(rain, f)
    forest_hydrographs[f] = runoff

    forest_results.append({
        "Infiltration capacity (mm/h)": f,
        "Total rainfall (mm)": np.sum(rain),
        "Canopy interception (mm)": np.sum(rain) * INTERCEPTION,
        "Total infiltration (mm)": np.sum(infiltration),
        "Total runoff depth (mm)": np.sum(runoff),
        "Maximum runoff intensity (mm/h)": np.max(runoff) * 60,
        "Total runoff volume (m³)": np.sum(runoff) / 1000 * FOREST_AREA
    })


# ------------------------------------------------------------
# 4. Plot 1: Runoff depthHydrograph
# ------------------------------------------------------------

colors = {
    50: "blue",
    75: "green",
    100: "red"
}

plt.figure(figsize=(15, 7))

# Original rainfall hyetograph
plt.plot(
    time, rain * 60,
    color="black",
    linestyle="--",
    linewidth=2.5,
    label="Original rainfall"
)

# Runoff hydrographs by infiltration capacity
for f in INFILTRATION_RATES:

    plt.plot(
        time,
        forest_hydrographs[f] * 60,
        color=colors[f],
        linewidth=2.2,
        label=f"Infiltration capacity {f} mm/h"
    )

plt.xlabel("Time (min)", fontsize=19)
plt.ylabel("Rainfall and runoff intensity (mm/h)", fontsize=19)
plt.title("Yashikirin: runoff hydrographs", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Envelope calculation function
# Maximum moving-window mean for each duration
# ------------------------------------------------------------

def calc_forest_envelope(wave, windows):

    wave = np.asarray(wave, dtype=float)
    cumulative = np.insert(np.cumsum(wave), 0, 0.0)

    envelope = np.zeros(len(windows))

    for i, T in enumerate(windows):

        moving_average = (
            cumulative[T:] - cumulative[:-T]
        ) / T

        envelope[i] = np.max(moving_average)

    return envelope


# ------------------------------------------------------------
# 6. Calculate envelopes and Δρ
# ------------------------------------------------------------

MAX_WINDOW = min(180, len(rain))
windows = np.arange(1, MAX_WINDOW + 1)

# Original rainfall envelope [mm/h]
E_rain = calc_forest_envelope(rain, windows) * 60

E_forest = {}
delta_rho_forest = {}

for f in INFILTRATION_RATES:

    E_forest[f] = calc_forest_envelope(
        forest_hydrographs[f], windows
    ) * 60

    # Difference between the maximum envelope values
    delta_rho_forest[f] = E_rain - E_forest[f]


# ------------------------------------------------------------
# 7. Plot 2: Δρ＋Original rainfall envelope
# ------------------------------------------------------------

plt.figure(figsize=(15, 7))

plt.plot(
    windows, E_rain,
    color="black",
    linestyle="--",
    linewidth=3,
    label="Original rainfall envelope"
)

for f in INFILTRATION_RATES:

    plt.plot(
        windows,
        delta_rho_forest[f],
        color=colors[f],
        linewidth=2.2,
        label=f"Δρ: Infiltration capacity {f} mm/h"
    )

plt.axhline(
    0,
    color="gray",
    linestyle=":",
    linewidth=1
)

plt.xlabel("Time window T (min)", fontsize=19)
plt.ylabel("Rainfall intensity / Δρ (mm/h)", fontsize=19)
plt.title("Yashikirin: Δρ envelope", fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(True, alpha=0.3)
plt.xlim(0, MAX_WINDOW)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. Results table
# ------------------------------------------------------------

forest_df = pd.DataFrame(forest_results)

print("===== Yashikirin runoff calculation results =====")
display(forest_df.round(3))

envelope_forest_df = pd.DataFrame({
    "Time window (min)": windows,
    "Original rainfall envelope(mm/h)": E_rain
})

for f in INFILTRATION_RATES:
    envelope_forest_df[
        f"Infiltration capacity{f}_Runoff envelope (mm/h)"
    ] = E_forest[f]

    envelope_forest_df[
        f"Infiltration capacity{f}_Δρ(mm/h)"
    ] = delta_rho_forest[f]

print("===== Yashikirin envelope results =====")
display(envelope_forest_df.head(10))

In [ ]:
# ============================================================
# Step 6: Rain garden area ratio and infiltration-capacity comparison
# Area ratios: 5, 10, 15, 20%
# Infiltration capacities: 50, 75, 100 mm/h
# Storage depth: 30 cm
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Calculation parameters
# ------------------------------------------------------------

AREA_RATIOS = [5, 10, 15, 20]   # Rain garden area ratio [%]
INFIL_RATES = [50, 75, 100]     # Infiltration capacity [mm/h]

CATCHMENT_AREA = 100.0         # Entire contributing catchment [m²]
RG_DEPTH = 0.30                 # Storage depth [m]

rain = np.asarray(r_1min, dtype=float)
time = np.arange(len(rain))

if not np.all(np.isfinite(rain)) or np.any(rain < 0):
    raise ValueError("r_1min contains invalid values")

colors = {
    5: "purple",
    10: "blue",
    15: "green",
    20: "red"
}

# ------------------------------------------------------------
# 2. Rain garden model
# ------------------------------------------------------------

def calc_rg_ratio(rain, area_ratio, infil_rate):

    rg_area = CATCHMENT_AREA * area_ratio / 100
    capacity = rg_area * RG_DEPTH

    infil_capacity = (
        infil_rate / 1000 / 60 * rg_area
    )

    storage = 0.0

    n = len(rain)
    runoff = np.zeros(n)
    depth = np.zeros(n + 1)
    infiltration = np.zeros(n)

    for i in range(n):

        # Rainfall inflow from the entire catchment
        vin = rain[i] / 1000 * CATCHMENT_AREA
        storage += vin

        # Infiltration
        vi = min(storage, infil_capacity)
        storage -= vi

        # Overflow
        vover = max(storage - capacity, 0.0)
        storage -= vover

        # Runoff depth normalized by the contributing catchment area
        runoff[i] = vover / CATCHMENT_AREA * 1000
        infiltration[i] = vi / CATCHMENT_AREA * 1000
        depth[i+1] = storage / rg_area

    return runoff, depth, infiltration


# ------------------------------------------------------------
# 3. Envelope calculation
# ------------------------------------------------------------

def calc_envelope(wave, windows):

    cumulative = np.insert(
        np.cumsum(np.asarray(wave, dtype=float)),
        0, 0.0
    )

    return np.array([
        np.max((cumulative[T:] - cumulative[:-T]) / T)
        for T in windows
    ])


MAX_WINDOW = min(180, len(rain))
windows = np.arange(1, MAX_WINDOW + 1)

E_rain = calc_envelope(rain, windows) * 60

# ------------------------------------------------------------
# 4. Calculate all cases
# ------------------------------------------------------------

rg_results = {}
delta_results = {}
result_table = []

for ratio in AREA_RATIOS:
    for f in INFIL_RATES:

        runoff, depth, infiltration = calc_rg_ratio(
            rain, ratio, f
        )

        E_runoff = calc_envelope(runoff, windows) * 60
        delta = E_rain - E_runoff

        rg_results[(ratio, f)] = runoff
        delta_results[(ratio, f)] = delta

        result_table.append({
            "Area ratio (%)": ratio,
            "Infiltration capacity (mm/h)": f,
            "Storage capacity (mm)": RG_DEPTH * 1000 * ratio / 100,
            "Maximum depth (cm)": np.max(depth) * 100,
            "Total runoff depth (mm)": np.sum(runoff),
            "Maximum runoff intensity (mm/h)": np.max(runoff) * 60,
            "Maximum Δρ (mm/h)": np.max(delta)
        })

# ------------------------------------------------------------
# 5. Plot 1: Hydrograph
# Compare area ratios at 75 mm/h infiltration capacity
# ------------------------------------------------------------

plt.figure(figsize=(15, 7))

plt.plot(
    time, rain * 60,
    "k--", linewidth=3,
    label="Original rainfall"
)

for ratio in AREA_RATIOS:
    plt.plot(
        time,
        rg_results[(ratio, 75)] * 60,
        color=colors[ratio],
        linewidth=2.2,
        label=f"Area ratio {ratio}%"
    )

plt.xlabel("Time (min)", fontsize=19)
plt.ylabel("Runoff intensity (mm/h)", fontsize=19)
plt.title("Rain garden: runoff response by area ratio (infiltration capacity: 75 mm/h)",
          fontsize=20)
plt.xticks(fontsize=17)
plt.yticks(fontsize=17)
plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Plot 2: Δρ envelope
# Compare area ratios for each infiltration capacity (three plots)
# ------------------------------------------------------------

for f in INFIL_RATES:

    plt.figure(figsize=(15, 7))

    plt.plot(
        windows, E_rain,
        "k--", linewidth=3,
        label="Original rainfall envelope"
    )

    for ratio in AREA_RATIOS:

        plt.plot(
            windows,
            delta_results[(ratio, f)],
            color=colors[ratio],
            linewidth=2.2,
            label=f"Area ratio {ratio}%"
        )

    plt.xlabel("Time window T (min)", fontsize=19)
    plt.ylabel("Rainfall intensity / Δρ (mm/h)", fontsize=19)
    plt.title(
        f"Rain garden: Δρ by area ratio(Infiltration capacity{f}mm/h)",
        fontsize=20
    )

    plt.xticks(fontsize=17)
    plt.yticks(fontsize=17)
    plt.legend(fontsize=16)
    plt.grid(alpha=0.3)
    plt.xlim(0, MAX_WINDOW)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 7. Plot 3: Infiltration-capacity comparison
# Compare area ratios of 10% and 15%
# ------------------------------------------------------------

infil_colors = {
    50: "blue",
    75: "green",
    100: "red"
}

for ratio in [10, 15]:

    plt.figure(figsize=(15, 7))

    plt.plot(
        windows, E_rain,
        "k--", linewidth=3,
        label="Original rainfall envelope"
    )

    for f in INFIL_RATES:

        plt.plot(
            windows,
            delta_results[(ratio, f)],
            color=infil_colors[f],
            linewidth=2.2,
            label=f"Infiltration capacity {f} mm/h"
        )

    plt.xlabel("Time window T (min)", fontsize=19)
    plt.ylabel("Rainfall intensity / Δρ (mm/h)", fontsize=19)
    plt.title(
        f"Rain garden: Δρ by infiltration capacity(Area ratio{ratio}%)",
        fontsize=20
    )
    plt.xticks(fontsize=17)
    plt.yticks(fontsize=17)
    plt.legend(fontsize=16)
    plt.grid(alpha=0.3)
    plt.xlim(0, MAX_WINDOW)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 8. Results table
# ------------------------------------------------------------

result_df = pd.DataFrame(result_table)

print("===== Rain garden: area ratio and infiltration capacity comparison =====")
display(result_df.round(3))

In [ ]:
# ============================================================
# A4 portrait, three columns, square panels
# (a) Schoolyard detention (b) Yashikirin (c) Rain garden
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

school_diams = [5, 10, 15, 20]
rg_ratios = [10, 15, 20]

school_colors = {
    5: "blue", 10: "green",
    15: "orange", 20: "red"
}
rg_colors = {
    10: "blue", 15: "green", 20: "red"
}

# A4 portrait, three columns
fig, axes = plt.subplots(
    1, 3,
    figsize=(8.27, 11.69),
    sharex=True,
    sharey=True
)

# (a) Schoolyard detention
ax = axes[0]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")

for d in school_diams:
    ax.plot(
        windows, delta_rho[d],
        color=school_colors[d], lw=1.5,
        label=f"{d} cm"
    )

ax.set_title("(a) Schoolyard storage", fontsize=11)
ax.set_ylabel(r"$\Delta\rho$ (mm/h)", fontsize=10)
ax.legend(fontsize=7, loc="upper right")

# (b) Yashikirin
ax = axes[1]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")
ax.plot(
    windows, delta_rho_forest[75],
    color="forestgreen", lw=1.8,
    label="75 mm/h"
)

ax.set_title("(b) Yashikirin", fontsize=11)
ax.legend(fontsize=7, loc="upper right")

# (c) Rain garden
ax = axes[2]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")

for ratio in rg_ratios:
    ax.plot(
        windows, delta_results[(ratio, 75)],
        color=rg_colors[ratio], lw=1.5,
        label=f"{ratio}%"
    )

ax.set_title("(c) Rain garden", fontsize=11)
ax.legend(fontsize=7, loc="upper right")

# Shared axis settings
for ax in axes:
    ax.set_box_aspect(1)  # Make each plotting panel square
    ax.set_xlim(0, 180)
    ax.set_xlabel("Time window (min)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(True, alpha=0.25)

# Adjust spacing between panels
fig.subplots_adjust(
    left=0.08,
    right=0.99,
    wspace=0.18
)

plt.show()

# Save figures
fig.savefig(
    "three_facilities_square_A4.png",
    dpi=300
)
fig.savefig(
    "three_facilities_square_A4.pdf"
)

In [ ]:
# ============================================================
# A4 portrait, three columns, square panels
# (a) Yashikirin (b) Schoolyard detention (c) Rain garden
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

school_diams = [5, 10, 15, 20]
forest_infil = [50, 75, 100]
rg_ratios = [10, 15, 20]

school_colors = {
    5: "blue", 10: "green",
    15: "orange", 20: "red"
}
forest_colors = {
    50: "blue", 75: "green", 100: "red"
}
rg_colors = {
    10: "blue", 15: "green", 20: "red"
}

# A4 portrait, three columns
fig, axes = plt.subplots(
    1, 3,
    figsize=(8.27, 11.69),
    sharex=True,
    sharey=True
)

# ------------------------------------------------------------
# (a) Yashikirin
# ------------------------------------------------------------
ax = axes[0]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")

for infil in forest_infil:
    ax.plot(
        windows, delta_rho_forest[infil],
        color=forest_colors[infil], lw=1.5,
        label=f"{infil} mm/h"
    )

ax.set_title("(a) Yashikirin", fontsize=11)
ax.set_ylabel(r"$\Delta\rho$ (mm/h)", fontsize=10)
ax.legend(fontsize=7, loc="upper right")

# ------------------------------------------------------------
# (b) Schoolyard detention
# ------------------------------------------------------------
ax = axes[1]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")

for d in school_diams:
    ax.plot(
        windows, delta_rho[d],
        color=school_colors[d], lw=1.5,
        label=f"{d} cm"
    )

ax.set_title("(b) Schoolyard storage", fontsize=11)
ax.legend(fontsize=7, loc="upper right")

# ------------------------------------------------------------
# (c) Rain garden
# ------------------------------------------------------------
ax = axes[2]
ax.plot(windows, E_rain, "k--", lw=2, label="Rainfall")

for ratio in rg_ratios:
    ax.plot(
        windows, delta_results[(ratio, 75)],
        color=rg_colors[ratio], lw=1.5,
        label=f"{ratio}%"
    )

ax.set_title("(c) Rain garden", fontsize=11)
ax.legend(fontsize=7, loc="upper right")

# Shared axis settings
for ax in axes:
    ax.set_box_aspect(1)
    ax.set_xlim(0, 180)
    ax.set_xlabel("Time window (min)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(True, alpha=0.25)

fig.subplots_adjust(
    left=0.08,
    right=0.99,
    wspace=0.18
)

plt.show()

fig.savefig("three_facilities_square_A4.png", dpi=300)
fig.savefig("three_facilities_square_A4.pdf")

In [ ]:
# ============================================================
# Summary of Δρ at a 60-minute time window
# (a) Yashikirin (b) Schoolyard detention (c) Rain garden
# ============================================================

import numpy as np

T_target = 60.0

# Use linear interpolation if 60 min is absent from windows
def value_at_60(values):
    return np.interp(T_target, windows, values)

print("=" * 55)
print("Time window = 60 min")
print("Δρ (mm/h)")
print("=" * 55)

# Rainfall
print(f"Rainfall                : {value_at_60(E_rain):8.3f}")

# (a) Yashikirin
print("\n(a) Yashikirin")
for infil in [50, 75, 100]:
    value = value_at_60(delta_rho_forest[infil])
    print(f"  {infil:3d} mm/h            : {value:8.3f}")

# (b) Schoolyard detention
print("\n(b) Schoolyard storage")
for d in [5, 10, 15, 20]:
    value = value_at_60(delta_rho[d])
    print(f"  {d:3d} cm              : {value:8.3f}")

# (c) Rain garden
print("\n(c) Rain garden")
for ratio in [10, 15, 20]:
    value = value_at_60(delta_results[(ratio, 75)])
    print(f"  {ratio:3d} %               : {value:8.3f}")

print("=" * 55)